# Section 1 — Movie Recommendation System
**ML Case Study 122 · B.Tech CSE Sem V**

| Model | Role |
|---|---|
| **KNN** | Main recommendation engine (finds similar movies) |
| **K-Means** | Movie grouping (segments) |
| **Hierarchical** | Movie grouping + dendrogram (tree diagram) |
| **Random Forest** | Predict whether a user will like a movie |

**Pipeline:** Data → Cleaning → EDA → Features → 4 Models → Analysis → Final KNN Recommender → Streamlit

# Section 2 — Problem Definition
| | |
|---|---|
| **Problem** | A movie platform wants to use movie data and user ratings to suggest movies to each user |
| **Objective** | Use the ratings users gave to suggest movies **for that user**, and also suggest movies **similar to a given movie** |
| **Input** | A user's rating history (for user suggestions) or a movie title (for similar movies); user history + movie details (for Random Forest) |
| **Output** | Top-N movies for a user · Top-N similar movies · movie groups (clusters) · chance that a user likes a movie |
| **ML approach** | KNN (similarity) · K-Means & Hierarchical (unsupervised) · Random Forest (supervised) |

**How the ratings are used:** (1) the movies a user *liked* (rating ≥ 4) are used as starting movies to find new suggestions, (2) some liked movies are hidden and used to check how good the suggestions are, (3) the rating history of users and movies is used as input for Random Forest.

**Target for Random Forest:** `like = 1` if `rating >= 4`, else `0` (made from the real ratings).

# Section 3 — Dataset & Documentation
- **File:** `movie_ratings_merged.csv` — one row per user rating, joined with movie details (ratings data + movie data).
- **Why this dataset:** it has *both* ratings (for personal suggestions) and movie details (genres, overview) → so we can do similarity, grouping and like-prediction.
- **Source:** `__________` *(write your dataset source / link here)*
- **Main columns:** `userId`, `movieId`, `rating` (ratings) · `title`, `genres`, `overview`, `release_year`, `vote_average`, `vote_count` (movie)
- **Limitations:** most users rated very few movies (matrix is mostly empty) · popular movies get most ratings · no user details like age or city.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.sparse import hstack
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, adjusted_rand_score, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

SEED = 42                       # fixed number so results are same every run
np.random.seed(SEED)
os.makedirs("models", exist_ok=True)

df = pd.read_csv("movie_ratings_merged.csv")

# check that all needed columns are present
needed_columns = ["userId", "movieId", "rating", "title", "genres", "overview",
                  "release_year", "vote_average", "vote_count"]
for col in needed_columns:
    assert col in df.columns, "Required column missing: " + col

missing_pct = (df.isnull().mean() * 100).round(2)
print("Shape:", df.shape, "| Users:", df.userId.nunique(), "| Movies:", df.movieId.nunique())
print("\nData types:\n", df.dtypes)
print("\nMissing % :\n", missing_pct[missing_pct > 0])
print("\nDuplicate rows:", df.duplicated().sum(), "| Duplicate (user, movie):", df.duplicated(["userId", "movieId"]).sum())
df.head(3)

# Section 4 — Data Cleaning
Fix data types → remove duplicates → fill missing values (median / 0 / empty text) → clean text & genres → create `like` target.

In [ ]:
rows_before = len(df)

# 1. correct data types (wrong values become NaN)
for col in ["rating", "release_year", "vote_average", "vote_count"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 2. remove rows where important columns are empty
df = df.dropna(subset=["userId", "movieId", "rating", "title"])

# 3. remove duplicates (keep the last rating if a user rated the same movie twice)
df = df.drop_duplicates()
df = df.drop_duplicates(["userId", "movieId"], keep="last")
df = df.reset_index(drop=True)
df[["userId", "movieId"]] = df[["userId", "movieId"]].astype(int)

# 4. fill missing values
df["release_year"] = df.release_year.fillna(df.release_year.median()).astype(int)
df["vote_average"] = df.vote_average.fillna(df.vote_average.median())
df["vote_count"] = df.vote_count.fillna(0)

# 5. clean genres:  "Sci-Fi|Drama" -> "SciFi Drama"
df["genres"] = df.genres.fillna("Unknown")
df["genres"] = df.genres.replace("(no genres listed)", "Unknown")
df["genres_clean"] = df.genres.str.replace("|", " ", regex=False)
df["genres_clean"] = df.genres_clean.str.replace(r"[^A-Za-z ]", "", regex=True)

# 6. clean overview text (remove symbols)
df["overview"] = df.overview.fillna("")
df["overview"] = df.overview.str.replace(r"[^A-Za-z0-9\s]", " ", regex=True)

# 7. target column: 1 if rating is 4 or more, else 0
df["like"] = (df.rating >= 4.0).astype(int)

print(f"Rows: {rows_before:,} -> {len(df):,}")
print(df["like"].value_counts(normalize=True).round(3))

# Section 5 — Exploratory Data Analysis
Rating distribution · movie popularity · top movies · genres · missing values · correlation.

In [ ]:
# one row per movie: how many ratings it got, its average rating, how often it was liked
by_movie = df.groupby("movieId")
movie_stats = pd.DataFrame()
movie_stats["n_ratings"] = by_movie["rating"].count()
movie_stats["mean_rating"] = by_movie["rating"].mean()
movie_stats["like_rate"] = by_movie["like"].mean()
movie_stats = movie_stats.reset_index()

movie_info = df.drop_duplicates("movieId").merge(movie_stats, on="movieId")

# count how many movies are in each genre
all_genres = []
for text in movie_info.genres_clean:
    for word in text.split():
        all_genres.append(word)
genre_counts = pd.Series(all_genres).value_counts()

print(f"User-item matrix density: {len(df) / (df.userId.nunique() * df.movieId.nunique()):.2%}")

fig, ax = plt.subplots(2, 3, figsize=(17, 9))

# plot 1: rating distribution
rating_counts = df.rating.value_counts().sort_index()
rating_counts.plot.bar(ax=ax[0, 0], color="teal", title="Rating distribution")

# plot 2: ratings per movie (popularity)
ax[0, 1].hist(movie_stats.n_ratings, bins=40, color="slateblue")
ax[0, 1].set(title="Ratings per movie (popularity)", yscale="log")

# plot 3: top 10 most rated movies
top10 = movie_info.nlargest(10, "n_ratings")
top10 = top10.set_index("title")["n_ratings"]
top10[::-1].plot.barh(ax=ax[0, 2], color="seagreen", title="Top 10 most-rated movies")   # [::-1] = reverse order

# plot 4: genre distribution
genre_counts.head(12)[::-1].plot.barh(ax=ax[1, 0], color="crimson", title="Genre distribution")

# plot 5: missing values in raw data
missing_pct[missing_pct > 0].plot.barh(ax=ax[1, 1], color="gray", title="Missing values in raw data (%)")

# plot 6: correlation heatmap
corr = movie_info[["n_ratings", "mean_rating", "vote_average", "vote_count", "release_year"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax[1, 2])
ax[1, 2].set_title("Correlation")

plt.tight_layout()
plt.show()

# Section 6 — Feature Engineering
| Technique | Used for | Why |
|---|---|---|
| **TF-IDF** (genres + overview) | KNN, clustering | converts words into numbers; common words get less weight |
| **Normalize** | KNN | makes every movie vector the same length, so cosine similarity compares content and not text length |
| **SVD (100 columns)** | clustering | shrinks thousands of TF-IDF columns to 100 columns, so K-Means / Ward work faster |
| **StandardScaler** | year, vote average, vote count | brings all numbers to the same scale for distance-based clustering |
| **Genre flags (0/1)** | Random Forest | simple 0/1 columns, trees do not need scaling |

In [ ]:
movies = movie_info[["movieId", "title", "genres", "genres_clean", "overview", "release_year", "vote_average",
                     "vote_count", "n_ratings", "mean_rating", "like_rate"]].reset_index(drop=True)
movies["title_key"] = movies.title.str.lower().str.strip()

# movieId -> row number in the feature matrix
movie_row = {}
for row_number, movie_id in enumerate(movies.movieId):
    movie_row[movie_id] = row_number

# ---- TF-IDF on genres and overview ----
genre_tfidf = TfidfVectorizer(token_pattern=r"\S+")
text_tfidf = TfidfVectorizer(stop_words="english", max_features=5000, min_df=2)
genre_matrix = genre_tfidf.fit_transform(movies.genres_clean)
text_matrix = text_tfidf.fit_transform(movies.overview)

# join genre columns and overview columns side by side, then normalize
joined = hstack([genre_matrix, text_matrix]).tocsr()
X_content = normalize(joined)

# ---- Features for clustering: SVD of content + scaled number columns ----
n_components = min(100, X_content.shape[1] - 1)
svd = TruncatedSVD(n_components=n_components, random_state=SEED)
Z = svd.fit_transform(X_content)
Z = normalize(Z)

numbers = np.column_stack([movies.release_year, movies.vote_average, np.log1p(movies.vote_count)])
numbers = StandardScaler().fit_transform(numbers)
X_cluster = np.hstack([Z, 0.3 * numbers])          # 0.3 = give numbers less weight than text

# ---- Genre flags for Random Forest (top 12 genres, 1 = movie has this genre) ----
top_genres = []
for g in genre_counts.index:
    if g != "Unknown" and len(top_genres) < 12:
        top_genres.append(g)

for g in top_genres:
    flags = []
    for text in movies.genres_clean:
        if g in text.split():
            flags.append(1)
        else:
            flags.append(0)
    movies["g_" + g] = flags

genre_cols = []
for g in top_genres:
    genre_cols.append("g_" + g)

print("Movies:", len(movies), "| Content matrix:", X_content.shape, "| Clustering matrix:", X_cluster.shape)

# Section 7 — Model 1: KNN (Main Recommender)
- **Goal:** find the movies most similar to a chosen movie (we search for neighbours, not classify).
- **How:** cosine distance on TF-IDF vectors; `similarity = 1 − distance`.
- **Version for a user:** take the neighbours of *every movie the user liked*, add up their similarity scores, hide the movies the user already rated → user's Top-N. This is how ratings drive the suggestions.
- **Experiment:** genres only vs overview only vs genres + overview (final).
- **Evaluation:** hide 20% of each user's liked movies, recommend using the rest, then measure **Precision@10, Recall@10, Hit-rate@10**; compared with *Random* and *Most-popular* suggestions.

In [ ]:
# ---- Test set: users with at least 6 liked movies. 80% of likes = starting movies, 20% = hidden ----
rng = np.random.default_rng(SEED)

liked_by_user = {}
for user, group in df[df.like == 1].groupby("userId"):
    movie_list = list(group["movieId"])
    if len(movie_list) >= 6:
        liked_by_user[user] = movie_list

eligible_users = list(liked_by_user.keys())
chosen_users = rng.choice(eligible_users, min(300, len(eligible_users)), replace=False)

splits = []          # each item = (starting movies, hidden movies)
for u in chosen_users:
    items = rng.permutation(liked_by_user[u])
    n_test = max(1, int(0.2 * len(items)))
    hidden_movies = items[:n_test]
    seed_movies = items[n_test:n_test + 20]
    splits.append((seed_movies, hidden_movies))


def build_knn(X):
    knn = NearestNeighbors(metric="cosine", algorithm="brute")
    knn.fit(X)
    return knn


def knn_recs(knn, X, seeds, k=10):
    # suggestions for a user: neighbours of all starting movies
    rows = []
    for m in seeds:
        rows.append(movie_row[m])
    dist, ind = knn.kneighbors(X[rows], n_neighbors=51)
    score = {}
    for d_row, i_row in zip(dist, ind):
        for d, i in zip(d_row, i_row):
            if i not in rows:
                score[i] = score.get(i, 0) + 1 - d        # add up similarity over all starting movies
    best = sorted(score, key=score.get, reverse=True)[:k]  # highest score first
    return set(movies.movieId.iloc[best])


popular_list = movies.sort_values("n_ratings", ascending=False).movieId.tolist()

def pop_recs(seeds, k=10):
    # simply the most-rated movies which are not in seeds
    seeds_set = set(seeds)
    result = []
    for m in popular_list:
        if m not in seeds_set:
            result.append(m)
    return set(result[:k])


def rand_recs(seeds, k=10):
    # random movies which are not in seeds
    others = np.setdiff1d(movies.movieId, seeds)
    return set(rng.choice(others, k, replace=False))


def evaluate(method, knn=None, X=None):
    total_p = 0
    total_r = 0
    total_h = 0
    for seeds, hidden in splits:
        if method == "knn":
            recs = knn_recs(knn, X, seeds)
        elif method == "popular":
            recs = pop_recs(seeds)
        else:
            recs = rand_recs(seeds)
        hits = len(recs & set(hidden))          # how many hidden movies we found
        total_p += hits / 10
        total_r += hits / len(hidden)
        if hits > 0:
            total_h += 1
    n = len(splits)
    return {"Precision@10": total_p / n, "Recall@10": total_r / n, "HitRate@10": total_h / n}


# ---- Experiment: which features work best for KNN? (final model = genres + overview) ----
FINAL = "KNN: genres + overview"
variants = {"KNN: genres only": normalize(genre_matrix),
            "KNN: overview only": normalize(text_matrix),
            FINAL: X_content}

knn_models = {}
for name in variants:
    knn_models[name] = build_knn(variants[name])
knn_model = knn_models[FINAL]

res = {}
for name in variants:
    res[name] = evaluate("knn", knn_models[name], variants[name])
res["Most popular"] = evaluate("popular")
res["Random"] = evaluate("random")

knn_results = pd.DataFrame(res).T.round(4)
print("Users evaluated:", len(splits))
print(knn_results)
print("Movies with empty overview:", (movies.overview.str.strip() == "").sum(), "of", len(movies))

# Example: neighbours of the most-rated movie
q = int(movies.n_ratings.idxmax())
dist, ind = knn_model.kneighbors(X_content[q], n_neighbors=6)
print("\nSimilar to:", movies.title[q])
similar = movies.iloc[ind[0][1:]][["title", "genres", "release_year"]].copy()
similar["similarity"] = (1 - dist[0][1:]).round(3)
similar

**Result analysis:** precision/recall are small for all methods because only the *hidden liked movies* count as correct. Compare (1) the three KNN versions → which features carry the signal, (2) KNN vs *Random* and *Most popular*. If *Most popular* is higher, that is a real finding: KNN on movie content is strong for finding similar movies and for new movies with no ratings, but popularity captures taste shared by most users. Movies with empty overview get weaker text similarity (limitation).

# Section 8 — Model 2: K-Means (Movie Grouping)
- **K-Means** divides movies into **K groups (clusters)**; K is chosen by us before running.
- **Choosing K:** Elbow (inertia) + Silhouette for K = 2…12; final K = best silhouette with **K ≥ 4** (K = 2 is too coarse).
- Cluster names come from the data (top genres/words), not typed by hand.

In [ ]:
def sil(X, labels):
    # silhouette score (higher = better separated clusters); uses a sample of movies for speed
    return silhouette_score(X, labels, sample_size=min(5000, len(X)), random_state=SEED)


ks = range(2, min(12, len(movies) - 1) + 1)

fits = {}
inertia_list = []
silhouette_list = []
for k in ks:
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_cluster)
    fits[k] = model
    inertia_list.append(model.inertia_)
    silhouette_list.append(sil(X_cluster, model.labels_))

scores = pd.DataFrame({"k": list(ks), "inertia": inertia_list, "silhouette": silhouette_list})

# best K = highest silhouette among K >= 4
good = scores[scores.k >= 4]
best_k = int(good.loc[good.silhouette.idxmax(), "k"])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(scores.k, scores.inertia, "o-")
ax[0].set(title="Elbow Method", xlabel="K", ylabel="Inertia")
ax[1].plot(scores.k, scores.silhouette, "o-r")
ax[1].set(title="Silhouette vs K", xlabel="K")
for a in ax:
    a.axvline(best_k, ls="--", c="gray")
plt.tight_layout()
plt.show()

kmeans = fits[best_k]
movies["kmeans_cluster"] = kmeans.labels_
km_sil = sil(X_cluster, kmeans.labels_)
print(f"Selected K = {best_k} | inertia = {kmeans.inertia_:.1f} | silhouette = {km_sil:.4f}")

In [ ]:
genre_words = genre_tfidf.get_feature_names_out()
text_words = text_tfidf.get_feature_names_out()
n_genre_cols = genre_matrix.shape[1]


def describe(labels, rows):
    # make a summary table: what is inside each cluster (uses average TF-IDF of its movies)
    out = []
    for c in np.unique(labels):
        member_rows = rows[labels == c]
        center = np.asarray(X_content[member_rows].mean(axis=0)).ravel()
        top_g = genre_words[center[:n_genre_cols].argsort()[::-1][:3]]      # 3 biggest genre values
        top_t = text_words[center[n_genre_cols:].argsort()[::-1][:5]]       # 5 biggest word values
        members = movies.iloc[member_rows]
        out.append({"cluster": c,
                    "size": len(member_rows),
                    "label": " / ".join(top_g[:2]),
                    "top_genres": ", ".join(top_g),
                    "top_words": ", ".join(top_t),
                    "median_year": int(members.release_year.median()),
                    "avg_user_rating": round(members.mean_rating.mean(), 2)})
    return pd.DataFrame(out)


# PCA = squeeze all columns into 2 columns, only to draw the clusters on a 2-D plot
pca = PCA(2, random_state=SEED).fit(X_cluster)

def plot2d(rows, labels, title):
    xy = pca.transform(X_cluster[rows])
    plt.figure(figsize=(7, 5))
    plt.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="tab20", s=10)
    plt.colorbar(label="cluster")
    plt.title(title)
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.show()


all_rows = np.arange(len(movies))
plot2d(all_rows, kmeans.labels_, f"K-Means clusters (K={best_k})")

km_profile = describe(kmeans.labels_, all_rows)
movies["cluster_label"] = movies.kmeans_cluster.map(km_profile.set_index("cluster").label)
km_profile

**Interpretation:** read each row — `top_genres` and `top_words` show what the cluster contains (give it your own name, e.g. "Drama / relationships"). The 2-D plot is only a squeezed picture, so clusters may overlap visually.

# Section 9 — Model 3: Hierarchical Clustering
| K-Means | Hierarchical |
|---|---|
| Divides data into **K groups**, K is fixed first | Builds a **tree** by joining the closest movies step by step, drawn as a **dendrogram** |
| K needed in advance | Cut the tree at any height |
| Works on all movies | Slow on big data → run on a **random sample** |

**Ward method:** at each step, join the two groups that increase the spread inside the group the least (same goal as K-Means, so results can be compared).

In [ ]:
# take a random sample of movies (hierarchical clustering is slow on big data)
n_h = min(1500, len(movies))
h_rows = np.random.default_rng(SEED).choice(len(movies), n_h, replace=False)
h_rows = np.sort(h_rows)
Xh = X_cluster[h_rows]

# build the tree (each row of L = one joining step; 3rd column = distance at which the join happened)
L = linkage(Xh, method="ward")

# height at which the tree gives exactly best_k clusters = middle of the last two joining distances
height_a = L[-(best_k - 1), 2]
height_b = L[-best_k, 2]
cut = (height_a + height_b) / 2

plt.figure(figsize=(11, 4.5))
dendrogram(L, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=9, color_threshold=cut)
plt.axhline(cut, ls="--", c="k")
plt.title(f"Dendrogram (Ward, {n_h} movies)")
plt.ylabel("Distance")
plt.show()

# cut the tree into best_k clusters (labels start from 1, so subtract 1)
h_labels = fcluster(L, t=best_k, criterion="maxclust") - 1

h_sil = silhouette_score(Xh, h_labels)
km_sil_same = silhouette_score(Xh, kmeans.labels_[h_rows])
ari = adjusted_rand_score(kmeans.labels_[h_rows], h_labels)      # ARI: how similar the 2 groupings are (1 = same)
print(f"Silhouette -> Hierarchical: {h_sil:.4f} | K-Means (same movies): {km_sil_same:.4f} | ARI between them: {ari:.3f}")

plot2d(h_rows, h_labels, f"Hierarchical clusters (Ward, K={best_k})")
describe(h_labels, h_rows)

**Interpretation:** tall branches join very different groups, short branches join almost identical movies; the dashed line is where the tree is cut. ARI shows how similar the K-Means and hierarchical groupings are. Here it is used only for *analysis*, not for recommending.

# Section 10 — Model 4: Random Forest (Like Prediction)
- **Target:** `like` (rating ≥ 4) — made from real ratings, nothing invented → **binary classification** (like / dislike).
- **Features:** user history, movie history, `vote_average`, `vote_count`, `release_year`, genre flags.
- **No cheating (no data leakage):** first split into train/test; history numbers are calculated from **train rows only**, and for a train row its own rating is left out of the average.
- **Limitation:** random split, not split by date.

In [ ]:
rf_df = df[["userId", "movieId", "rating", "like"]].merge(
    movies[["movieId", "vote_average", "vote_count", "release_year"] + genre_cols], on="movieId")
rf_df = rf_df.sample(min(len(rf_df), 300_000), random_state=SEED)       # limit rows for speed on huge data

train, test = train_test_split(rf_df, test_size=0.2, random_state=SEED, stratify=rf_df["like"])


def history(train, test, key, p):
    # For each user (or movie) calculate: number of ratings, average rating, like-rate.
    # Numbers come from TRAIN data only.
    global_rating = train["rating"].mean()      # used when a user/movie has no history
    global_like = train["like"].mean()

    stats = train.groupby(key).agg(total_rating=("rating", "sum"), total_like=("like", "sum"),
                                   n_given=("rating", "size"))

    # --- train rows: leave out the row's own rating (so the model cannot see its own answer) ---
    t = train[[key, "rating", "like"]].join(stats, on=key)
    others = t["n_given"] - 1                         # number of OTHER ratings
    others_safe = others.replace(0, np.nan)           # avoid divide by zero
    train_feat = pd.DataFrame(index=train.index)
    train_feat[p + "_cnt"] = others
    train_feat[p + "_mean"] = ((t["total_rating"] - t["rating"]) / others_safe).fillna(global_rating)
    train_feat[p + "_like"] = ((t["total_like"] - t["like"]) / others_safe).fillna(global_like)

    # --- test rows: use the full train history ---
    e = test[[key]].join(stats, on=key)
    test_feat = pd.DataFrame(index=test.index)
    test_feat[p + "_cnt"] = e["n_given"].fillna(0)
    test_feat[p + "_mean"] = (e["total_rating"] / e["n_given"]).fillna(global_rating)
    test_feat[p + "_like"] = (e["total_like"] / e["n_given"]).fillna(global_like)

    return train_feat, test_feat


u_tr, u_te = history(train, test, "userId", "u")
m_tr, m_te = history(train, test, "movieId", "m")

meta = ["vote_average", "vote_count", "release_year"] + genre_cols
X_tr = pd.concat([u_tr, m_tr, train[meta]], axis=1)
X_te = pd.concat([u_te, m_te, test[meta]], axis=1)
y_tr = train["like"]
y_te = test["like"]

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_te)                    # 0 or 1
proba = rf.predict_proba(X_te)[:, 1]       # chance of like

majority = max(y_te.mean(), 1 - y_te.mean())     # accuracy if we always guess the bigger class
rf_metrics = {"Accuracy": accuracy_score(y_te, pred),
              "Precision": precision_score(y_te, pred),
              "Recall": recall_score(y_te, pred),
              "F1": f1_score(y_te, pred),
              "ROC-AUC": roc_auc_score(y_te, proba)}
print(f"Majority-class baseline accuracy: {majority:.4f}")
rounded = {}
for name in rf_metrics:
    rounded[name] = round(rf_metrics[name], 4)
print(rounded)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["Dislike", "Like"], cmap="Blues", ax=ax[0])
importance = pd.Series(rf.feature_importances_, index=X_tr.columns).sort_values()
importance.tail(10).plot.barh(ax=ax[1], color="darkgreen")
ax[0].set_title("Confusion matrix (test)")
ax[1].set_title("Top 10 feature importances")
plt.tight_layout()
plt.show()

# Error analysis: is the model worse for users with little history?
print("\nError rate by user-history size (quartiles):")
is_wrong = (pred != y_te)
history_groups = pd.qcut(X_te["u_cnt"], 4, duplicates="drop")      # 4 equal-size groups by history size
print(is_wrong.groupby(history_groups).mean().round(3))

**Interpretation:** accuracy must be compared with the **majority-class baseline**; precision, recall, F1 and ROC-AUC show how well the *Like* class is predicted. The importance ranking is only a rough guide. The error table shows whether predictions are weaker for users with little history (new users).

# Section 11 — Model Comparison
| Model | Type | Purpose | Output | Evaluation |
|---|---|---|---|---|
| **KNN** | Similarity based | Similar movies / recommendation | Top-N movies | Precision@K, Recall@K, Hit-rate@K |
| **K-Means** | Unsupervised | Movie grouping | Clusters | Inertia, Elbow, Silhouette |
| **Hierarchical** | Unsupervised | Similarity / grouping | Dendrogram + clusters | Silhouette, dendrogram |
| **Random Forest** | Supervised (many trees) | Like prediction | Class + chance of like | Accuracy, P, R, F1, AUC, Confusion matrix |

> The models solve **different tasks**, so they are *not* ranked by one accuracy number.

In [ ]:
knn_text = (f"Precision@10 = {knn_results.loc[FINAL, 'Precision@10']}, "
            f"Recall@10 = {knn_results.loc[FINAL, 'Recall@10']}, "
            f"HitRate@10 = {knn_results.loc[FINAL, 'HitRate@10']}")
km_text = f"K = {best_k}, silhouette = {km_sil:.4f}, inertia = {kmeans.inertia_:.1f}"
hier_text = f"K = {best_k}, silhouette = {h_sil:.4f}, ARI vs K-Means = {ari:.3f}"
rf_text = (f"Acc = {rf_metrics['Accuracy']:.3f} (majority = {majority:.3f}), "
           f"F1 = {rf_metrics['F1']:.3f}, AUC = {rf_metrics['ROC-AUC']:.3f}")

pd.DataFrame([["KNN", knn_text],
              ["K-Means", km_text],
              ["Hierarchical", hier_text],
              ["Random Forest", rf_text]],
             columns=["Model", "Actual results from this run"])

# Section 12 — Final Recommendation Logic
**KNN is the final recommendation engine** because it:
1. directly gives a ranked Top-N list (similar movies, or suggestions for a user based on the movies they liked),
2. works for new movies with no ratings — only genres/overview are needed,
3. is easy to explain (shared genres/words + similarity score),
4. is checked with recommendation metrics (Precision@K / Recall@K).

**K-Means & Hierarchical** = supporting analysis (movie groups). **Random Forest** = like-prediction experiment.

# Section 13 — Final Recommendation Function
`recommend_movies(title, top_n)` and `recommend_for_user(user_id, top_n)` are saved in `recommender.py`, so the **notebook and Streamlit use the same code**. Steps: find the movie → take its feature vector → KNN finds the nearest movies → return Top-N. If the title is wrong, a friendly message with suggestions is shown.

In [ ]:
%%writefile recommender.py
import difflib
import joblib
import numpy as np
import pandas as pd

_cache = {}          # remembers the loaded file, so it is not loaded again and again


def load_bundle(path="models/recommender_bundle.joblib"):
    if path not in _cache:
        _cache[path] = joblib.load(path)
    return _cache[path]


def find_movie(query, bundle=None):
    """Return (row number, suggestions). Exact title match first, then partial match; most-rated movie wins."""
    if bundle is None:
        bundle = load_bundle()
    movies = bundle["movies"]
    q = str(query).lower().strip()
    keys = movies["title_key"]

    # 1. exact match
    exact = movies[keys == q]
    if len(exact) > 0:
        return int(exact["n_ratings"].idxmax()), []

    # 2. partial match (query is part of the title)
    if q != "":
        partial = movies[keys.str.contains(q, regex=False)]
        if len(partial) > 0:
            return int(partial["n_ratings"].idxmax()), []

    # 3. not found: suggest close titles
    close = difflib.get_close_matches(q, keys.tolist(), n=5, cutoff=0.5)
    suggestions = []
    for c in close:
        suggestions.append(movies.loc[keys == c, "title"].iloc[0])
    return None, suggestions


def recommend_movies(movie_title, top_n=10, bundle=None):
    """Top-N movies most similar to movie_title (KNN, cosine similarity)."""
    if bundle is None:
        bundle = load_bundle()
    idx, suggestions = find_movie(movie_title, bundle)
    if idx is None:
        message = f"Movie '{movie_title}' not found."
        if len(suggestions) > 0:
            message = message + " Did you mean: " + "; ".join(suggestions)
        print(message)
        return pd.DataFrame()

    dist, ind = bundle["knn_model"].kneighbors(bundle["movie_matrix"][idx], n_neighbors=int(top_n) + 1)
    keep = ind[0] != idx                                   # remove the movie itself from its own list
    dist = dist[0][keep][:top_n]
    ind = ind[0][keep][:top_n]

    recs = bundle["movies"].iloc[ind][["title", "genres", "release_year", "vote_average", "n_ratings"]].copy()
    recs["similarity"] = (1 - dist).round(4)
    recs.insert(0, "Rank", np.arange(1, len(recs) + 1))
    return recs.reset_index(drop=True)


def recommend_for_user(user_id, top_n=10, bundle=None):
    """Top-N movies for a user: neighbours of the movies the user liked (rating >= 4), already-rated movies hidden."""
    if bundle is None:
        bundle = load_bundle()
    seeds = bundle["liked"].get(user_id, [])[-20:]          # last 20 liked movies
    if len(seeds) == 0:
        print(f"User {user_id} not found or has no liked movies.")
        return pd.DataFrame()

    dist, ind = bundle["knn_model"].kneighbors(bundle["movie_matrix"][seeds], n_neighbors=51)
    score = {}
    for d_row, i_row in zip(dist, ind):
        for d, i in zip(d_row, i_row):
            if i not in bundle["seen"][user_id]:
                score[i] = score.get(i, 0) + 1 - d

    top = sorted(score, key=score.get, reverse=True)[:int(top_n)]    # highest score first
    recs = bundle["movies"].iloc[top][["title", "genres", "release_year", "vote_average"]].copy()
    scores_list = []
    for i in top:
        scores_list.append(round(score[i], 3))
    recs["score"] = scores_list
    recs.insert(0, "Rank", np.arange(1, len(recs) + 1))
    return recs.reset_index(drop=True)

In [ ]:
# Save everything the Streamlit app needs
export_cols = ["movieId", "title", "title_key", "genres", "release_year", "vote_average", "n_ratings", "cluster_label"]

# user -> row numbers of liked movies
liked_rows = {}
for user, group in df[df.like == 1].groupby("userId"):
    liked_rows[user] = [movie_row[m] for m in group["movieId"]]

# user -> row numbers of all rated movies
seen_rows = {}
for user, group in df.groupby("userId"):
    seen_rows[user] = set([movie_row[m] for m in group["movieId"]])

bundle = {"knn_model": knn_model, "movie_matrix": X_content, "movies": movies[export_cols],
          "liked": liked_rows, "seen": seen_rows}
joblib.dump(bundle, "models/recommender_bundle.joblib", compress=3)
joblib.dump({"model": rf, "features": list(X_tr.columns)}, "models/random_forest.joblib")

from recommender import recommend_movies, recommend_for_user, find_movie
print(recommend_movies("this movie does not exist", 5))          # wrong title is handled with a friendly message

# Section 14 — Final Results
Sample similar-movie and user recommendations. If a title is not in this dataset, the function says so (nothing is invented); the two most-rated movies are added so there is always some output.

In [ ]:
test_titles = ["Interstellar", "Inception", "The Dark Knight", "Avengers"] + movies.nlargest(2, "n_ratings").title.tolist()

for title in test_titles:
    idx, _ = find_movie(title)
    if idx is None:
        recommend_movies(title, 5)
        continue
    print(f"\n=== {title} -> {movies.title[idx]} | segment: {movies.cluster_label[idx]} ===")
    print(recommend_movies(title, 5).to_string(index=False))

# Recommendation for the user with the most liked movies
u = None
for user in liked_rows:
    if u is None or len(liked_rows[user]) > len(liked_rows[u]):
        u = user
print(f"\n=== Personalised: user {u} (liked {len(liked_rows[u])} movies, e.g. {movies.title[liked_rows[u][:3]].tolist()}) ===")
print(recommend_for_user(u, 5).to_string(index=False))

### Streamlit app
Run: `streamlit run app.py` (needs the `models/` folder and `recommender.py` in the same folder).

In [ ]:
%%writefile app.py
import streamlit as st
from recommender import load_bundle, find_movie, recommend_movies, recommend_for_user

st.set_page_config(page_title="Movie Recommender", layout="wide")
st.title("🎬 Movie Recommendation System")
st.caption("KNN on TF-IDF content features · personalised from users' liked movies")

bundle = load_bundle()
movies = bundle["movies"]

tab1, tab2 = st.tabs(["Similar movies", "Personalised for a user"])

with tab1:
    choice = st.selectbox("Search or select a movie", sorted(movies["title"].unique()))
    top_n = st.slider("Number of recommendations", 5, 20, 10)
    if st.button("Recommend"):
        row_number = find_movie(choice, bundle)[0]
        info = movies.loc[row_number]
        st.write(f"**{info['title']}** · {info['genres']} · segment: *{info['cluster_label']}*")
        st.dataframe(recommend_movies(choice, top_n, bundle), hide_index=True)

with tab2:
    user_id = st.selectbox("User ID", sorted(bundle["liked"]))
    n_user = st.slider("Number of recommendations", 5, 20, 10, key="n_user")
    if st.button("Recommend for user"):
        st.caption(f"Based on {len(bundle['liked'][user_id])} movies this user liked (rating ≥ 4)")
        st.dataframe(recommend_for_user(user_id, n_user, bundle), hide_index=True)

# Section 15 — Conclusion
- **Achieved:** complete pipeline — cleaning, EDA, features, 4 models, saved files, Streamlit app.
- **Best for recommendation:** **KNN** (only model that gives a ranked list, and can also suggest for a user from the movies they liked); see the Precision@K table for KNN versions vs baselines.
- **Clustering:** K-Means gives flat movie groups; Hierarchical shows how groups join together (dendrogram). Both are supporting analysis.
- **Random Forest:** predicts like/dislike from real ratings, checked against a majority baseline.
- **Limitations:** KNN on content only ignores who rated what · only known ratings used for evaluation · random (not date-based) split · hierarchical clustering run on a sample.
- **Future work:** collaborative filtering / hybrid model · sentence-embedding text features · cast & director features · date-based evaluation.

---
**Removed models:** Logistic Regression, Decision Tree, Naive Bayes, SVM, Gradient Boosting
**Final models:** KNN → recommendation · K-Means → grouping · Hierarchical → similarity/grouping · Random Forest → like prediction
**Architecture:** Dataset → Cleaning → EDA → Feature Engineering → KNN → K-Means → Hierarchical → Random Forest → Analysis → Final KNN Recommender → Streamlit